# Complaint2Concern
## Cross-validated triage acuity modeling with structured intake + chief complaint text

This notebook uses cross-validation (instead of a single split) to compare models and quantify safety-relevant behavior.


## Clinical framing and leakage guardrails
- Triage acuity is predicted from **triage-time** information only.
- Post-triage outcomes (for example `disposition`, `ed_los_hours`) are excluded.
- Undertriage (predicting less urgent than true acuity) is treated as a key safety metric.


In [ ]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix, hstack
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score

# Resolve project root
candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = None
for c in candidates:
    if (c / 'src').exists() and (c / 'configs').exists():
        PROJECT_ROOT = c
        break
if PROJECT_ROOT is None:
    raise RuntimeError('Could not locate PROJECT_ROOT with src/ and configs/.')

sys.path.append(str(PROJECT_ROOT))
print('PROJECT_ROOT:', PROJECT_ROOT)


In [ ]:
from src.config import load_config
from src.data_io import load_train_test
from src.embeddings import encode_complaints
from src.features_structured import fit_transform_structured, infer_structured_columns
from src.features_text import fit_transform_tfidf
from src.utils import make_classifier, set_seed


In [ ]:
cfg = load_config(PROJECT_ROOT / 'configs/minilm_catboost.yaml')
SEED = cfg['train'].get('seed', 42)
set_seed(SEED)

CV_MODE = 'stratified'   # 'stratified' or 'group_site'
N_SPLITS = 5
DEV_SAMPLE_N = None      # e.g., 20000 for fast local iteration

train_df, test_df = load_train_test(cfg)
if DEV_SAMPLE_N is not None and DEV_SAMPLE_N < len(train_df):
    train_df = train_df.sample(DEV_SAMPLE_N, random_state=SEED).reset_index(drop=True)

id_col = cfg['data']['id_col']
target_col = cfg['data']['target_col']
text_col = cfg['data']['text_col']
leakage_cols = cfg['data'].get('leakage_columns', [])

print('train shape:', train_df.shape)
print('test shape :', test_df.shape)
print('target col :', target_col)
print('text col   :', text_col)
print('null text train:', round(train_df[text_col].isna().mean(), 4))
print('null text test :', round(test_df[text_col].isna().mean(), 4))


## Quick triage-focused EDA


In [ ]:
print('Acuity distribution:')
display(train_df[target_col].value_counts(normalize=True).sort_index())

vitals = [c for c in ['heart_rate','respiratory_rate','temperature_c','spo2','systolic_bp','diastolic_bp','news2_score'] if c in train_df.columns]
if vitals:
    display(train_df.groupby(target_col)[vitals].median())

display(train_df[[id_col, text_col, target_col]].head(5))


## CV helpers


In [ ]:
label_encoder = LabelEncoder()
y_enc = label_encoder.fit_transform(train_df[target_col].astype(str))

classes_int = np.asarray(label_encoder.classes_).astype(int)
high_mask = classes_int <= 2


def decode_to_int(y_encoded: np.ndarray) -> np.ndarray:
    return label_encoder.inverse_transform(y_encoded.astype(int)).astype(int)


def eval_esi(y_true_enc: np.ndarray, y_pred_enc: np.ndarray) -> dict:
    y_true = decode_to_int(y_true_enc)
    y_pred = decode_to_int(y_pred_enc)
    high = y_true <= 2
    return {
        'accuracy': float(np.mean(y_true_enc == y_pred_enc)),
        'macro_f1': float(f1_score(y_true_enc, y_pred_enc, average='macro')),
        'off_by_1': float(np.mean(np.abs(y_true - y_pred) <= 1)),
        'undertriage_rate': float(np.mean(y_pred > y_true)),
        'severe_undertriage_rate': float(np.mean((y_pred - y_true) >= 2)),
        'recall_esi_1_2': float(np.mean(y_pred[high] <= 2)) if np.any(high) else np.nan,
    }


def make_splitter(df: pd.DataFrame, y: np.ndarray, mode: str, n_splits: int, seed: int):
    if mode == 'group_site' and 'site_id' in df.columns:
        gkf = GroupKFold(n_splits=n_splits)
        return list(gkf.split(df, y, groups=df['site_id']))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return list(skf.split(df, y))


## Cross-validated model comparison
Models:
1. Structured-only
2. Text-only TF-IDF
3. Structured + TF-IDF
4. Text-only MiniLM embeddings
5. Structured + MiniLM embeddings (fusion)


In [ ]:
splits = make_splitter(train_df, y_enc, mode=CV_MODE, n_splits=N_SPLITS, seed=SEED)
print(f'CV mode={CV_MODE}, folds={len(splits)}')

n = len(train_df)
n_classes = len(label_encoder.classes_)

# OOF containers for innovation and safety analyses
oof = {
    'true_enc': y_enc.copy(),
    'pred_structured': np.full(n, -1, dtype=int),
    'pred_text_tfidf': np.full(n, -1, dtype=int),
    'pred_structured_tfidf': np.full(n, -1, dtype=int),
    'pred_text_emb': np.full(n, -1, dtype=int),
    'pred_fusion': np.full(n, -1, dtype=int),
    'proba_structured': np.zeros((n, n_classes), dtype=float),
    'proba_text_emb': np.zeros((n, n_classes), dtype=float),
    'proba_fusion': np.zeros((n, n_classes), dtype=float),
}

rows = []

emb_model_name = cfg['embeddings'].get('model_name', 'sentence-transformers/all-MiniLM-L6-v2')
emb_batch_size = cfg['embeddings'].get('batch_size', 64)
emb_backend = cfg['embeddings'].get('backend', 'sentence_transformers')

for fold, (tr_idx, va_idx) in enumerate(splits, start=1):
    t0 = time.time()

    tr = train_df.iloc[tr_idx]
    va = train_df.iloc[va_idx]
    y_tr = y_enc[tr_idx]
    y_va = y_enc[va_idx]

    struct_cols = infer_structured_columns(
        tr,
        target_col=target_col,
        text_col=text_col,
        id_col=id_col,
        explicit_drop_cols=leakage_cols,
    )
    struct_pre, x_tr_struct, x_va_struct = fit_transform_structured(tr[struct_cols], va[struct_cols])

    # 1) Structured-only
    model_struct = make_classifier(
        model_name='catboost',
        params={'iterations': 350, 'depth': 6, 'learning_rate': 0.05, 'verbose': False},
        n_classes=n_classes,
    )
    model_struct.fit(x_tr_struct, y_tr)
    pred_struct = np.asarray(model_struct.predict(x_va_struct)).reshape(-1).astype(int)
    proba_struct = np.asarray(model_struct.predict_proba(x_va_struct))

    # 2) Text-only TF-IDF + LR
    tfidf_vec, x_tr_tfidf, x_va_tfidf = fit_transform_tfidf(
        tr[text_col], va[text_col], max_features=25000, ngram_range=(1, 2)
    )
    model_tfidf = LogisticRegression(max_iter=1200, solver='lbfgs')
    model_tfidf.fit(x_tr_tfidf, y_tr)
    pred_tfidf = model_tfidf.predict(x_va_tfidf).astype(int)

    # 3) Structured + TF-IDF + LR
    x_tr_struct_sparse = csr_matrix(x_tr_struct)
    x_va_struct_sparse = csr_matrix(x_va_struct)
    x_tr_struct_tfidf = hstack([x_tr_struct_sparse, x_tr_tfidf]).tocsr()
    x_va_struct_tfidf = hstack([x_va_struct_sparse, x_va_tfidf]).tocsr()
    model_struct_tfidf = LogisticRegression(max_iter=1200, solver='lbfgs')
    model_struct_tfidf.fit(x_tr_struct_tfidf, y_tr)
    pred_struct_tfidf = model_struct_tfidf.predict(x_va_struct_tfidf).astype(int)

    # 4) Text-only MiniLM embeddings
    x_tr_emb = encode_complaints(
        tr[text_col],
        model_name=emb_model_name,
        batch_size=emb_batch_size,
        backend=emb_backend,
        normalize_embeddings=True,
    )
    x_va_emb = encode_complaints(
        va[text_col],
        model_name=emb_model_name,
        batch_size=emb_batch_size,
        backend=emb_backend,
        normalize_embeddings=True,
    )

    model_emb = make_classifier(
        model_name='lightgbm',
        params={'n_estimators': 350, 'learning_rate': 0.05, 'num_leaves': 63},
        n_classes=n_classes,
    )
    model_emb.fit(x_tr_emb, y_tr)
    pred_emb = np.asarray(model_emb.predict(x_va_emb)).reshape(-1).astype(int)
    proba_emb = np.asarray(model_emb.predict_proba(x_va_emb))

    # 5) Structured + MiniLM fusion
    x_tr_fusion = np.hstack([x_tr_struct, x_tr_emb])
    x_va_fusion = np.hstack([x_va_struct, x_va_emb])
    model_fusion = make_classifier(
        model_name='catboost',
        params={'iterations': 450, 'depth': 6, 'learning_rate': 0.05, 'verbose': False},
        n_classes=n_classes,
    )
    model_fusion.fit(x_tr_fusion, y_tr)
    pred_fusion = np.asarray(model_fusion.predict(x_va_fusion)).reshape(-1).astype(int)
    proba_fusion = np.asarray(model_fusion.predict_proba(x_va_fusion))

    # Save OOF
    oof['pred_structured'][va_idx] = pred_struct
    oof['pred_text_tfidf'][va_idx] = pred_tfidf
    oof['pred_structured_tfidf'][va_idx] = pred_struct_tfidf
    oof['pred_text_emb'][va_idx] = pred_emb
    oof['pred_fusion'][va_idx] = pred_fusion
    oof['proba_structured'][va_idx] = proba_struct
    oof['proba_text_emb'][va_idx] = proba_emb
    oof['proba_fusion'][va_idx] = proba_fusion

    model_preds = {
        'structured_only': pred_struct,
        'text_tfidf_only': pred_tfidf,
        'structured_plus_tfidf': pred_struct_tfidf,
        'text_emb_only': pred_emb,
        'structured_plus_emb_fusion': pred_fusion,
    }

    for model_name, pred in model_preds.items():
        rows.append({
            'fold': fold,
            'model': model_name,
            **eval_esi(y_va, pred),
            'fold_runtime_sec': round(time.time() - t0, 2),
        })

    print(f'Fold {fold}/{len(splits)} done in {round(time.time()-t0,2)}s')

cv_results_df = pd.DataFrame(rows)
display(cv_results_df.head())


In [ ]:
cv_summary = (
    cv_results_df
    .groupby('model', as_index=False)
    .agg(
        folds=('fold', 'count'),
        accuracy_mean=('accuracy', 'mean'),
        accuracy_std=('accuracy', 'std'),
        macro_f1_mean=('macro_f1', 'mean'),
        macro_f1_std=('macro_f1', 'std'),
        undertriage_mean=('undertriage_rate', 'mean'),
        severe_undertriage_mean=('severe_undertriage_rate', 'mean'),
        recall_esi_1_2_mean=('recall_esi_1_2', 'mean'),
    )
    .sort_values('macro_f1_mean', ascending=False)
)
display(cv_summary)


## OOF innovation layer: text-vs-structure concern gap
Uses out-of-fold predictions to reduce optimistic bias.


In [ ]:
oof_df = train_df[[id_col, text_col]].copy()
oof_df['true_acuity'] = decode_to_int(oof['true_enc'])
oof_df['pred_structured'] = decode_to_int(oof['pred_structured'])
oof_df['pred_text_emb'] = decode_to_int(oof['pred_text_emb'])
oof_df['pred_fusion'] = decode_to_int(oof['pred_fusion'])

oof_df['p_high_text'] = oof['proba_text_emb'][:, high_mask].sum(axis=1)
oof_df['p_high_struct'] = oof['proba_structured'][:, high_mask].sum(axis=1)
oof_df['text_concern_gap'] = oof_df['p_high_text'] - oof_df['p_high_struct']

gap_threshold = np.quantile(oof_df['text_concern_gap'], 0.90)
alerts = oof_df[oof_df['text_concern_gap'] >= gap_threshold]

print('gap threshold (p90):', round(float(gap_threshold), 4))
print('alert case count:', len(alerts))
print('alert true high-acuity rate:', round((alerts['true_acuity'] <= 2).mean(), 4))
display(alerts.sort_values('text_concern_gap', ascending=False).head(15))


## Review-flag logic (OOF)


In [ ]:
fusion_conf = oof['proba_fusion'].max(axis=1)
low_conf_threshold = np.quantile(fusion_conf, 0.10)
high_gap_threshold = np.quantile(oof_df['text_concern_gap'], 0.90)

oof_df['fusion_confidence'] = fusion_conf
oof_df['review_flag'] = (
    (oof_df['fusion_confidence'] <= low_conf_threshold)
    | (oof_df['text_concern_gap'] >= high_gap_threshold)
)

flagged = oof_df[oof_df['review_flag']]
print('review flag rate:', round(len(flagged) / len(oof_df), 4))
print('high-acuity rate among flagged:', round((flagged['true_acuity'] <= 2).mean(), 4))


## Subgroup snapshot (OOF fusion predictions)


In [ ]:
subgroups = [c for c in ['age_group', 'sex', 'site_id'] if c in train_df.columns]
sub_rows = []

for col in subgroups:
    temp = pd.DataFrame({
        col: train_df[col].values,
        'y_true': decode_to_int(oof['true_enc']),
        'y_pred': decode_to_int(oof['pred_fusion']),
    })
    for grp, gdf in temp.groupby(col):
        if len(gdf) < 50:
            continue
        sub_rows.append({
            'group_col': col,
            'group_val': grp,
            'n': len(gdf),
            'accuracy': float(np.mean(gdf['y_true'] == gdf['y_pred'])),
            'undertriage_rate': float(np.mean(gdf['y_pred'] > gdf['y_true'])),
        })

subgroup_df = pd.DataFrame(sub_rows).sort_values(['group_col', 'n'], ascending=[True, False])
display(subgroup_df.head(30))


## Final submission via CV fold-ensemble
We train the selected model on each fold and average test probabilities.


In [ ]:
SELECTED_MODEL = 'structured_plus_emb_fusion'  # choose from compared models

splits = make_splitter(train_df, y_enc, mode=CV_MODE, n_splits=N_SPLITS, seed=SEED)
proba_test_accum = np.zeros((len(test_df), len(label_encoder.classes_)), dtype=float)

for fold, (tr_idx, va_idx) in enumerate(splits, start=1):
    tr = train_df.iloc[tr_idx]
    y_tr = y_enc[tr_idx]

    struct_cols = infer_structured_columns(
        tr,
        target_col=target_col,
        text_col=text_col,
        id_col=id_col,
        explicit_drop_cols=leakage_cols,
    )
    struct_pre, x_tr_struct, x_test_struct = fit_transform_structured(tr[struct_cols], test_df[struct_cols])

    if SELECTED_MODEL == 'structured_only':
        model = make_classifier('catboost', {'iterations': 350, 'depth': 6, 'learning_rate': 0.05, 'verbose': False}, n_classes=len(label_encoder.classes_))
        model.fit(x_tr_struct, y_tr)
        proba_test = np.asarray(model.predict_proba(x_test_struct))

    elif SELECTED_MODEL == 'text_tfidf_only':
        vec, x_tr_tfidf, x_test_tfidf = fit_transform_tfidf(tr[text_col], test_df[text_col], max_features=25000, ngram_range=(1, 2))
        model = LogisticRegression(max_iter=1200, solver='lbfgs')
        model.fit(x_tr_tfidf, y_tr)
        proba_test = np.asarray(model.predict_proba(x_test_tfidf))

    elif SELECTED_MODEL == 'structured_plus_tfidf':
        vec, x_tr_tfidf, x_test_tfidf = fit_transform_tfidf(tr[text_col], test_df[text_col], max_features=25000, ngram_range=(1, 2))
        x_tr_mix = hstack([csr_matrix(x_tr_struct), x_tr_tfidf]).tocsr()
        x_test_mix = hstack([csr_matrix(x_test_struct), x_test_tfidf]).tocsr()
        model = LogisticRegression(max_iter=1200, solver='lbfgs')
        model.fit(x_tr_mix, y_tr)
        proba_test = np.asarray(model.predict_proba(x_test_mix))

    elif SELECTED_MODEL == 'text_emb_only':
        x_tr_emb = encode_complaints(tr[text_col], model_name=emb_model_name, batch_size=emb_batch_size, backend=emb_backend, normalize_embeddings=True)
        x_test_emb = encode_complaints(test_df[text_col], model_name=emb_model_name, batch_size=emb_batch_size, backend=emb_backend, normalize_embeddings=True)
        model = make_classifier('lightgbm', {'n_estimators': 350, 'learning_rate': 0.05, 'num_leaves': 63}, n_classes=len(label_encoder.classes_))
        model.fit(x_tr_emb, y_tr)
        proba_test = np.asarray(model.predict_proba(x_test_emb))

    elif SELECTED_MODEL == 'structured_plus_emb_fusion':
        x_tr_emb = encode_complaints(tr[text_col], model_name=emb_model_name, batch_size=emb_batch_size, backend=emb_backend, normalize_embeddings=True)
        x_test_emb = encode_complaints(test_df[text_col], model_name=emb_model_name, batch_size=emb_batch_size, backend=emb_backend, normalize_embeddings=True)
        x_tr_mix = np.hstack([x_tr_struct, x_tr_emb])
        x_test_mix = np.hstack([x_test_struct, x_test_emb])
        model = make_classifier('catboost', {'iterations': 450, 'depth': 6, 'learning_rate': 0.05, 'verbose': False}, n_classes=len(label_encoder.classes_))
        model.fit(x_tr_mix, y_tr)
        proba_test = np.asarray(model.predict_proba(x_test_mix))

    else:
        raise ValueError(f'Unsupported SELECTED_MODEL: {SELECTED_MODEL}')

    proba_test_accum += proba_test
    print(f'Ensemble fold {fold}/{len(splits)} complete')

proba_test_mean = proba_test_accum / len(splits)
test_pred_enc = proba_test_mean.argmax(axis=1)
test_pred = label_encoder.inverse_transform(test_pred_enc).astype(int)

submission = pd.DataFrame({id_col: test_df[id_col], target_col: test_pred})
out_dir = PROJECT_ROOT / 'outputs'
out_dir.mkdir(parents=True, exist_ok=True)
submission_path = out_dir / 'submission_notebook_cv.csv'
submission.to_csv(submission_path, index=False)
print('saved:', submission_path)
display(submission.head())


## Deployment concept
- Use model output as triage support, not clinician replacement.
- Show acuity suggestion + confidence.
- Trigger review when concern gap is high or confidence is low.
